# CampusHub · Analyse des données et des modèles

Notebook du volet Data (cahier des charges §2.1) : estimation du prix des livres, recommandations, et indicateurs d'usage de la plateforme.

Toutes les données personnelles restent dans MongoDB : seules des agrégations sont manipulées ici.

**Lancer :** `docker compose run --rm -p 8888:8888 ml sh -c "pip install -r requirements-notebook.txt && jupyter lab --ip 0.0.0.0 --allow-root --no-browser"`

In [ ]:
import os, sys, json
sys.path.insert(0, '..')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pymongo import MongoClient

import pricing
from recommend import evaluate, rank

plt.rcParams.update({'figure.figsize': (8, 4), 'axes.spines.top': False, 'axes.spines.right': False, 'font.size': 11})
ACCENT = '#2547D0'
MONGO_URL = os.environ.get('MONGO_URL', 'mongodb://mongo:27017/campushub?directConnection=true')

## 1. Estimation du prix d'un livre d'occasion

Comparaison de trois modèles sur un jeu de test tenu à l'écart (20 %) : la **moyenne** (baseline imposée par le cahier), la **régression linéaire**, puis la **Random Forest**.

In [ ]:
metrics = pricing.train(save=False)
scores = pd.DataFrame({k: v for k, v in metrics.items() if k in ('baseline_moyenne', 'regression_lineaire', 'random_forest')}).T
scores

In [ ]:
ax = scores['mae'].plot.barh(color=['#CDD2DA', '#A9BBF2', ACCENT])
ax.set_xlabel('Erreur absolue moyenne (DT)')
ax.set_title('MAE par modèle (plus bas = meilleur)')
for i, v in enumerate(scores['mae']):
    ax.text(v + 0.1, i, f'{v:.2f} DT', va='center')
plt.tight_layout()

### Données d'entraînement

Faute de données réelles en début de projet (risque identifié au §7), le modèle s'entraîne sur 2 400 annonces synthétiques, complétées par les annonces publiées sur la plateforme. Le générateur encode des effets non linéaires que la régression linéaire ne capture qu'en partie.

In [ ]:
data = pricing.make_synthetic()
data['ratio'] = data['prix'] / data['prix_neuf']
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
data.groupby('age')['ratio'].mean().plot(ax=axes[0], color=ACCENT, marker='o')
axes[0].set(title='Décote moyenne selon l’âge', xlabel='Âge du livre (années)', ylabel='Prix / prix neuf')
data.groupby('etat_rank')['ratio'].mean().rename(index=dict(enumerate(pricing.ETATS))).plot.bar(ax=axes[1], color=ACCENT)
axes[1].set(title='Ratio moyen selon l’état', xlabel='', ylabel='Prix / prix neuf')
plt.tight_layout()

In [ ]:
model = pricing.load_model()
names = model.named_steps['pre'].get_feature_names_out()
importance = pd.Series(model.named_steps['model'].feature_importances_, index=[n.split('__')[-1] for n in names]).sort_values().tail(10)
importance.plot.barh(color=ACCENT, title='Importance des variables (Random Forest)')
plt.tight_layout()

### Fourchette conseillée

La fourchette affichée aux vendeurs correspond à l'écart interquartile des prédictions des 200 arbres : plus les arbres sont en désaccord, plus la fourchette est large.

In [ ]:
examples = [('Probabilités', 'TRES_BON', 2021, 45), ('Python', 'NEUF', 2024, 60), ('Anglais technique', 'ACCEPTABLE', 2012, 45)]
pd.DataFrame([{ 'matière': m, 'état': e, 'année': a, 'prix neuf': p, **pricing.predict_range(model, m, e, a, p)} for m, e, a, p in examples])

## 2. Recommandations de groupes et d'événements

Filtrage par contenu : TF-IDF sur le texte des groupes/événements et sur le profil de l'étudiant (matières pondérées ×2, centres d'intérêt, filière), puis similarité cosinus. Évaluation par **précision@k** face à une baseline de **popularité**.

In [ ]:
rows = []
for k in (1, 3, 5, 10):
    r = evaluate(k=k)
    rows.append({'k': k, 'TF-IDF': r[f'precision_at_{k}'], 'Popularité': r['baseline_popularite']})
prec = pd.DataFrame(rows).set_index('k')
ax = prec.plot(marker='o', color=[ACCENT, '#CDD2DA'], title='Précision@k')
ax.set_ylim(0, 1)
prec

> Les étudiants synthétiques choisissent leurs matières parmi un catalogue où chaque groupe porte une matière explicite : la précision réelle sera plus basse. À refaire avec les adhésions réelles une fois la plateforme utilisée par la promotion (critère de validation : au moins 20 testeurs).

## 3. Indicateurs d'usage de la plateforme

Agrégations directement dans MongoDB : aucun nom ni e-mail ne sort de la base.

In [ ]:
try:
    db = MongoClient(MONGO_URL, serverSelectionTimeoutMS=2000).get_default_database()
    db.command('ping')
except Exception as e:
    db = None
    print('MongoDB indisponible :', e)

In [ ]:
if db is not None:
    actifs = pd.DataFrame(db.activites.aggregate([{'$group': {'_id': '$jour', 'utilisateurs': {'$sum': 1}}}, {'$sort': {'_id': 1}}]))
    if len(actifs):
        actifs.set_index('_id')['utilisateurs'].plot.bar(color=ACCENT, title='Utilisateurs actifs par jour')
        plt.tight_layout()
    else:
        print('Pas encore d’activité enregistrée.')

In [ ]:
if db is not None:
    par_matiere = pd.DataFrame(db.annonces.aggregate([{'$group': {'_id': '$matiere', 'annonces': {'$sum': 1}, 'prix_moyen': {'$avg': '$prix'}}}, {'$sort': {'annonces': -1}}, {'$limit': 10}]))
    trajets = pd.DataFrame(db.trajets.aggregate([{'$group': {'_id': '$lieuDepart', 'trajets': {'$sum': 1}, 'places': {'$sum': '$placesTotales'}}}, {'$sort': {'trajets': -1}}, {'$limit': 10}]))
    display(par_matiere.rename(columns={'_id': 'matière'}), trajets.rename(columns={'_id': 'départ'}))